<a href="https://colab.research.google.com/github/EnasIbrahimAli2005/Spam-and-Ham-Email-Classification-Project/blob/main/Spam_Ham_Emails(Enas_Ibrahiem).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas textblob


In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
!kaggle datasets download -d abdallahwagih/spam-emails

Dataset URL: https://www.kaggle.com/datasets/abdallahwagih/spam-emails
License(s): apache-2.0
100% 207k/207k [00:00<00:00, 384kB/s]
100% 207k/207k [00:00<00:00, 383kB/s]


In [ ]:
!unzip spam-emails.zip

Archive:  spam-emails.zip
  inflating: spam.csv                


In [ ]:
import pandas as pd
from textblob import TextBlob
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample
import pandas as pd
from textblob import TextBlob

import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample

In [ ]:
data = pd.read_csv('spam.csv', delimiter='\t', header=None, names=['Category', 'Message'])

In [ ]:
data

,Category,Message
0,"Category,Message",NaN
1,"ham,""Go until jurong point, crazy.. Available ...",NaN
2,"ham,Ok lar... Joking wif u oni...",NaN
3,"spam,Free entry in 2 a wkly comp to win FA Cup...",NaN
4,"ham,U dun say so early hor... U c already then...",NaN
...,...,...
5570,"spam,""This is the 2nd time we have tried 2 con...",NaN
5571,"ham,Will ü b going to esplanade fr home?",NaN
5572,"ham,""Pity, * was in mood for that. So...any ot...",NaN
5573,"ham,The guy did some bitching but I acted like...",NaN


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Load the dataset
data = pd.read_csv('spam.csv', delimiter=',', header=0, names=['Category', 'Message'])

# Drop any rows where 'Message' is NaN
data.dropna(subset=['Message'], inplace=True)

# Function to clean text: remove numbers, punctuation, and convert to lowercase
def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = text.lower()
    text_blob = TextBlob(text)

    cleaned_text = " ".join(text_blob.split())
    return cleaned_text


In [ ]:
# Clean the 'Message' column
data['Message'] = data['Message'].apply(clean_text)


In [ ]:
# Separate 'ham' and 'spam' classes
ham_df = data[data['Category'] == 'ham']
spam_df = data[data['Category'] == 'spam']

In [ ]:
# Balance the dataset by undersampling 'ham' messages to match the number of 'spam' messages
if len(spam_df) > 0 and len(ham_df) > 0:
    ham_df_balanced = resample(ham_df, replace=False, n_samples=len(spam_df), random_state=42)
    df_balanced = pd.concat([ham_df_balanced, spam_df])
else:
    df_balanced = data  # Use original data if balancing is not possible

In [ ]:
# Feature extraction
tfidf_vectorizer = TfidfVectorizer()
X = tfidf_vectorizer.fit_transform(df_balanced['Message'])


In [ ]:
# Encode labels ('ham' as 0, 'spam' as 1)
y = df_balanced['Category'].apply(lambda x: 1 if x.lower() == 'spam' else 0)


In [ ]:
# Split the data into training and testing sets with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# Initialize the classifier (Naive Bayes)
classifier = MultinomialNB()

In [ ]:
# Train the classifier
classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred = classifier.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy * 100:.2f}%")

# Classification report
unique_classes = sorted(y_test.unique())
class_names = ['ham', 'spam']
class_names = [class_names[i] for i in unique_classes]

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names, labels=unique_classes))


Model Accuracy: 94.98%

Classification Report:
              precision    recall  f1-score   support

         ham       0.95      0.95      0.95       150
        spam       0.95      0.95      0.95       149

    accuracy                           0.95       299
   macro avg       0.95      0.95      0.95       299
weighted avg       0.95      0.95      0.95       299

